# Control NKT

Simple notebook for manual monitoring of laser power and temperature, to easily export curve over set period of time

| Device | Register | Description | Data Type |
| --- | --- | --- | --- |
| Harmonik-GW | 0x9D | Output power (W) | 32-bit float |
| Harmonik-GW | 0xBA | Temperature regulator temp (°C) | 32-bit float |
| Boostik HP | 0x1F | Output power (W) | 16-bit unsigned |
| Adjustik | 0x11 | Module temperature (m°C) | 16-bit signed |

In [1]:
import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from datetime import datetime
from pathlib import Path

In [16]:
from utils.naming import make_run_dir

In [14]:
import os
from sys import path
import struct
import time

In [3]:
path.append(r"D:/NKT Photonics/Examples/DLL_Example_Python")

In [4]:
from NKTP_DLL import *

Loading x64 DLL from: D:\NKT Photonics\\NKTPDLL\x64\NKTPDLL.dll


#### Scan for device

In [5]:
ports = getAllPorts()

print("Scanning:", ports)

result = openPorts(ports, 1, 1)
print("Open result:", PortResultTypes(result))

open_ports = getOpenPorts()

if open_ports:
    print("Device(s) found on:", open_ports)
else:
    print("No NKT devices found")

closePorts('')

Scanning: COM1,COM4,COM5
Open result: 0:OPSuccess
Device(s) found on: COM4


0

In [8]:
def debug_register_read(port='COM4', addr=70, reg=0x9D):
    """Debug function to read a register and print raw bytes."""
    openPorts(port, 0, 0)  # Open port

    # Read register
    result, raw_bytes = registerRead(port, addr, reg, -1)
    print(f"Register {hex(reg)} at address {addr}:")
    print(f"  Result: {result} (0 = success)")
    print(f"  Raw bytes: {raw_bytes} (length: {len(raw_bytes)})")

    closePorts(port)
    return result, raw_bytes

In [10]:

# Test Harmonik-GW power register (0x9D)
debug_register_read('COM4', addr=70, reg=0x9D)

# Test Adjustik temperature register (0x11)
debug_register_read('COM4', addr=128, reg=0x11)

Register 0x9d at address 70:
  Result: 7 (0 = success)
  Raw bytes: b'' (length: 0)
Register 0x11 at address 128:
  Result: 0 (0 = success)
  Raw bytes: b'L.' (length: 2)


(0, b'L.')

### Discover modules

In [8]:
def scan_modules(port):
    """Scan for NKT modules on a given port and return their addresses and types."""
    # Open the port
    openResult = openPorts(port, 0, 0)
    if openResult != 0:
        print(f"Failed to open {port}: {PortResultTypes(openResult)}")
        return []

    modules = []
    for addr in range(1, 161):  # Scan addresses 1-160
        # Read module type (register 0x61)
        module_type_result = registerReadU8(port, addr, 0x61, -1)
        if module_type_result[0] == 0:  # Success
            module_type = module_type_result[1]
            modules.append((addr, module_type))

    # Close the port
    closePorts(port)
    return modules

In [9]:
# Scan COM4
modules = scan_modules('COM4')
print("Found modules (address, type):", modules)

Found modules (address, type): [(1, 51), (63, 56), (64, 58), (74, 59), (80, 53), (96, 106), (128, 52)]


### Read Power and Temperature

In [11]:
def read_power_and_temp(port='COM4', harmonik_addr=74, adjustik_addr=128):
    """Read power (W) and temperature (°C) from NKT modules."""
    openPorts(port, 0, 0)  # Open port

    # Read power (Harmonik-GW, register 0x9D, float32)
    _, power_bytes = registerRead(port, harmonik_addr, 0x9D, -1)
    power = struct.unpack('f', power_bytes[:4])[0]  # Ensure 4 bytes for float32

    # Read temperature (Adjustik, register 0x11, I16 in m°C)
    _, temp_bytes = registerRead(port, adjustik_addr, 0x11, -1)
    temp = struct.unpack('h', temp_bytes[:2])[0] / 1000  # Ensure 2 bytes for I16

    closePorts(port)  # Close port
    return power, temp

#### Single test

In [12]:
# Fetch and display data
power, temp = read_power_and_temp()
print(f"Power: {power:.3f} W | Temperature: {temp:.2f} °C")

Power: 5.021 W | Temperature: 11.88 °C


In [15]:
print("Time (s) | Power (W) | Temperature (°C)")
for i in range(10):
    power, temp = read_power_and_temp()
    print(f"{i:.1f}      | {power:.3f}    | {temp:.2f}")
    time.sleep(0.1)

Time (s) | Power (W) | Temperature (°C)
0.0      | 4.988    | 11.87
1.0      | 4.987    | 11.89
2.0      | 4.986    | 11.89
3.0      | 4.981    | 11.89
4.0      | 4.980    | 11.89
5.0      | 4.978    | 11.89
6.0      | 4.972    | 11.88
7.0      | 4.971    | 11.89
8.0      | 4.967    | 11.87
9.0      | 4.966    | 11.85


#### Time acquisition

In [17]:
DATA_DIR = "data/nkt"
os.makedirs(DATA_DIR, exist_ok=True)

In [18]:
MEAS = "nkt_output"
SAMPLE = "harmonik"
ACQ_CELL = f"{MEAS}_{SAMPLE}"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

print("Temperature scan data will be saved in:", RUN_DIR)

Saving data to: data/nkt\nkt_output\2026-06-17_harmonik_001
Temperature scan data will be saved in: data/nkt\nkt_output\2026-06-17_harmonik_001


In [ ]:
T = 10 * 60 # 10 minutes
dt = 0.1 # time steps

In [ ]:
# Initialize the acquisition manager
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)



